# Basic XML Parsing: Starting Out with MODS

:::{important} Learning Outcomes in this Section
- A better understanding of MODS as a metadata scheme
- Importing and using the Beautiful Soup library
- Loading and Processing XML with Beautiful Soup
- Useful skills to examine metadata encoded in XML
:::

The [Metadata Object Description Schema](https://www.loc.gov/standards/mods/), 
more commonly known by its acronym MODS, has been used since its development in 2002 by libraries and other cultural heritage organizations to record and transfer information
about digital objects, bibliographic items, and other sorts of collection materials [@guentheretal2003].
While DublinCore was designed for describing any kind of digital content,
MODS was specifically intended to be a web-friendly scheme for bibliograpic data, particularly the MaRC standard. While MODS is much simpler than the MaRC structure, it is 
more complex (and possibly less widely used) than the DublinCore schema.
MODS may be encouuntered when working
with digital content maintained by libraries and archives, since
it is designed to be harmonious with the most widely used metadata approaches used in these types of organizations, and it is usually presented in XML.

As we will see in the more advanced XML usage sections,
the capacity to "mix and match" various schemes
is enabled by namespacing. This is frequently found in the descriptive sections of EAD records, which can contain MODS elements; the Smithsonian's Field Book project, likewise, uses the MODS schema in a "fusion" with two others, the Natural Collections Description (NCD) and Encoded Archival Context (EAC) [@sheffieldetal2011].

## Sample MODS Records

The book's data files include sample MODS records from the Library of Congress Web Archives (LCWA). This program uses MODS to describe and present a searchable catalog for the sites archived by the Library of Congress. You will find five sample MODS records in the data file `data/lcwa-mods-5.xml` (see [](#mods-ex01)).


```{code} xml
:label: mods-ex01
:filename: /data/lcwa-mods-5.xml (simplified)
:linenos:
:emphasize-lines: 11, 13
:caption: This excerpt shows brief snippets from the first few lines of the sample MODS collection from the LCWA. The full file can be found in the `data` directory. Note that some content was removed for readability, so the excerpt is not directly equivalent to the source.
<?xml version="1.0" encoding="UTF-8"?>
<modsCollection>
  <mods>
    <identifier>lcwaN0010234</identifier>
      <titleInfo>
        <title>Slate Magazine</title>
      </titleInfo>
    <language>
      <languageTerm authority="iso639-2b" type="code">eng</languageTerm>
    </language>
    . . .
  </mods>
. . . 
</modsCollection>
```

A brief look at the sample file (see [](#mods-ex01)) reveals that the root element of this file is `modsCollection`,
which wraps multiple full MODS records.
The file is formatted with each record on a single line, which is convenient for computers to read, but it makes it difficult for humans to read. To get a quick understanding and take a quick look at the full file, read it in using the Python-based Beautiful Soup library, which offers useful tools for parsing marked up text, both HTML and XML. Subsequent examples demonstrate XML-specific tools, which are needed to undertake more complex tasks like namespace processing, validation, editing, and writing. 

## A First Look with Beautiful Soup

[BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) is a python library designed for working with HTML.
XML, as noted earlier, is based on the same markup principles,
and BeautifulSoup offers a lightweight set of tools that can assist in basic parsing for both HTML and XML.

### Set Up: Loading MODS Data and Beautiful Soup

This section shows how to import BeautifulSoup in Python,
then how to load your data.  We will be using an additional library called `lxml`, which helps BeautifulSoup (aka BS4) to search and build XML. It is possible that you may need to do an extra step to install `lxml` if you have not used it before, and those steps are [outlined in the BS4 documentation here](https://beautiful-soup-4.readthedocs.io/en/latest/index.html?highlight=namespace#installing-a-parser).

:::{attention} Data Import
As throughout the book, all data files can be found in the `data` directory
which accompanies this book and can be found in book's GitHub repository.
This structure, which more or less packages the data alongside the worked examples in this notebook,
allows you to run the notebook yourself if you download the full repository from GitHub,
while using the same commands and producing the same results. 

The following cell illustrates a consistent way to reference the data files
without using complex paths later on.
:::

In [1]:
from pathlib import Path

DATA = Path('..', 'data')
OUT = Path('..', 'output')

MODS_collection = DATA / 'lcwa-mods-5.xml'

Now, import the BeautifulSoup library.

In [2]:
from bs4 import BeautifulSoup

## Load and Parse the MODS XML

Now that BeautifulSoup is ready, it's time to load the XML data from the `lcwa-mods-5.xml` file.
The following cell opens that XML file and assigns it to a reusable Python variable named `file` (line 1).
Then, using the `BeautifulSoup()` function, Python parses the data into the `metadata` variable (line 3).

In [3]:
file = open(MODS_collection, 'r')

metadata = BeautifulSoup(file, 'lxml-xml')

:::{hint} What is Parsing?
When you import a data file, like the MODS records encoded in XML, _parse_ is a fancy
way to refer to opening the file in a way that allows the computer to understand
the file in a particular way.
In this case, the BeautifulSoup library is "parsing" the received data as marked up
text. Because you know this is XML that follows the MODS standard,
you can then ask Python to process the data in certain ways.
If you didn't tell Python how to parse the file, it would just see it as
raw data without being able to name the various elements and data inside it.

We want to be able to refer to specific parts of the data by their element names,
which requires this parsing step.
:::

Now that the file is loaded and parsed, you can use BeautifulSoup to
look at it more. In BeautifulSoup, this data is often called the "soup,"
but to follow our metadata theme, the above cell names our soup `metadata`.

### Confirming the Data is Loaded

Let's take a look at the data. BeautifulSoup allows you to
see it as basic text, which can be a good way to make sure you have
indeed loaded the metadata that you are seeking.
You can use a slice to just see the first few characters.
To do this, you can append the `.text` method to your data variable (i.e., `metadata.text` as below), which returns the values inside the elements.
As a first step, this does not require knowledge of MODS, but if you look at [](#modes-ex01) above, you should see that the returned data is indeed the contents of the first element:

In [4]:
print(metadata.text[:100])


lcwaN001023485999109353Slate Magazineengelectronictext/htmlborn digitalgeneraltextweb siteUnited St


Notice that the response does not include any of the XML element names or attributes.
The response is a string of text from in between the tags of the MODS fields now available as the `metadata` object. 
Look closely at demonstrates that, yes,
this is the metadata!

## Navigating the Data

Now that the data is parsed and you have isolated a single record,
let's use BeautifulSoup to take a look further through the data.
As discussed in the next section, you can think of any XML element as addressable
by a path, but looking through an XML object like this is often
referred to as traversing the "tree".

BeautifulSoup allows for quick navigation through the tree with the `.content` method
and a basic "dot notation" that uses element names.
To get quick look at the structure of a tag, use the `.contents`. The next cell uses this tag to assign the contents of the `modsCollection` element to a new variable, which can then be revealed with the `.contents`.

Notice that the response is a list (it is printed with elements separated by commas `,` and enclosed in a set of square brackets `[ ]`).
Keep this in mind since this data property can be used later in another operation if you want to loop through the contents.

In [5]:
modsCollection_tag = metadata.modsCollection

modsCollection_tag.contents

['\n',
 <mods version="3.4" xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.loc.gov/mods/v3 http://www.loc.gov/standards/mods/v3/mods-3-4.xsd"><identifier>lcwaN0010234</identifier><identifier invalid="yes" type="database id">85999</identifier><identifier invalid="yes" type="database id">109353</identifier><titleInfo><title>Slate Magazine</title></titleInfo><language><languageTerm authority="iso639-2b" type="code">eng</languageTerm></language><physicalDescription><form authority="marcform">electronic</form><internetMediaType>text/html</internetMediaType><digitalOrigin>born digital</digitalOrigin></physicalDescription><targetAudience>general</targetAudience><typeOfResource>text</typeOfResource><genre authority="marcgt">web site</genre><originInfo><place><placeTerm type="text">United States</placeTerm></place></originInfo><abstract/><relatedItem type="host"><titleInfo><title>

The above illustrates the literalness of XML.
Because parsed XML makes everything identified within the tree a piece of addressable and processable data,
blank space that is not enclosed in tags may be treated as part of the tree.
This can initially be a confusing or frustrating, particularly if you are more interested in the readable contents (i.e., _the metadata_),
but it is an important feature of the metadata _as data_, which you can readily work around as long as you are aware of this feature.

### Using "Dot Notation"

If you are familiar with HTML and CSS selectors,
you may know that HTML `class` attributes (e.g., `<p class="body">A body paragraph</p>`)
can be "selected" in CSS using _dot notation_. In other words,
to apply a CSS style to the above, the CSS `.body {font-style: italic}`
would direct a browser to render the HTML paragraph in italic text.
In this system, the `.body` in the CSS "selects" any HTML tag with the `class="body"` attribute,
then renders that element's text in italics.
BeautifulSoup offers a similar dot notation that can be used to "select" markup elements.

Thus, individual elements in the sample MODS XML can located or "selected" using dot notation.
The notation can be used to call any element that you know is in the metadata,
as well as certain properties using built in BeautifulSoup methods to access the data, including
an element's `.text` (the string data between the element tags), its `.name` (the element tag name),
or its `.contents` (the whole element, its attributes, and its contents).
The sample file contains five full MODS records.
It may therefore be useful to isolate each of those.
To look at the metadata and pull out the first record that matches the element name,
you can navigate down in dot notation by appending tags.
For example, the next cell pulls up the first `titleInfo` element in the first `mods` element
found in the `modsCollection` element.

In [6]:
metadata.modsCollection.mods.titleInfo

<titleInfo><title>Slate Magazine</title></titleInfo>

### Extracting Data with Dot Notation

The dot notation can also be useful with BeautifulSoup's built in methods. For example, say you want to get the text of a title tag. You can request the dot notation, which looks through the descendant tags, and then append the `.text` method, which returns a string with the contents of the tag.

In [7]:
first_title = metadata.modsCollection.mods.titleInfo.title.text
print(type(first_title), first_title)

<class 'str'> Slate Magazine


### Basic Looping to Examine Elements

You can see that the `modsCollection` root contains multiple `mods` subelements, or children.
BeautifulSoup objects with child elements can be called with `.children` method, which
provides an interable list that can be called in a `for` loop.
In the following example, dot notation is used to assign the `modsCollection` element to `modsCollection_tag`.
Then, the `for` loop iterates through the sublements, isolating each individual MODS record.
Finally, filtering out blank elements, the loop prints the root tag (in this case, each one is `<mods>`)
and then uses dot notation to print the first `title` element found in the mods record.
Thus quickly displaying the name of the archived websites referenced in each metadata record.

In [8]:
modsCollecion_tag = metadata.modsCollection

for mods in modsCollecion_tag.children:
    if mods.name != None:
        print(mods.name, mods.title)

mods <title>Slate Magazine</title>
mods <title>Raw Story</title>
mods <title>Huffington Post</title>
mods <title>BuzzFeed</title>
mods <title>Drudge Report</title>


The above cell shows how to filter out blank elements on line 4
by filtering out any blank lines (elements without tags that display as `None` values in the contents list).
This behavior was revealed earlier with the `.contents` tag.

To look further at the MODS records, try looking for other known elements,
like `identifier`, `subject`, etc. Dot notation is only available in BeautifulSoup
and offers a quicker way of naming objects than the fully qualified namespaces that
we will see later when using the `lxml` library.

```{exercise} Count the mods elements
:label: ex-bs4-count-mods
How many MODS records are in this collection?
```

:::{tip} Count the `mods` elements
You could add a counter to the above to count the `mods` elements.
This may not be necessary with a small collection, but it provides a 
quick sanity check and also would be easier for larger collections.
:::

Unhide the following cell to see sample code with a counter added:

In [9]:
counter = 0 

modsCollecion_tag = metadata.modsCollection

for mods in modsCollecion_tag.children:
    if mods.name != None:
        counter += 1

print(f'Counted non-blank mods elements: {counter}')

Counted non-blank mods elements: 5


```{solution} ex-bs4-count-mods
:label: sol-bs4-count-mods
:class: dropdown
There are five `mods` elements in the file, which means there are five MODS records in this collection.
```

### Using `.find()` to Pull an Element

The following code illustrates how to display the first record in the MODS collection using the `.find()` function. If you know the name of the element you are seeking, you can identify it by name. In this case the function only returns the first match that it finds.

In [10]:
first_mods = metadata.find('mods')

print(first_mods)

<mods version="3.4" xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.loc.gov/mods/v3 http://www.loc.gov/standards/mods/v3/mods-3-4.xsd"><identifier>lcwaN0010234</identifier><identifier invalid="yes" type="database id">85999</identifier><identifier invalid="yes" type="database id">109353</identifier><titleInfo><title>Slate Magazine</title></titleInfo><language><languageTerm authority="iso639-2b" type="code">eng</languageTerm></language><physicalDescription><form authority="marcform">electronic</form><internetMediaType>text/html</internetMediaType><digitalOrigin>born digital</digitalOrigin></physicalDescription><targetAudience>general</targetAudience><typeOfResource>text</typeOfResource><genre authority="marcgt">web site</genre><originInfo><place><placeTerm type="text">United States</placeTerm></place></originInfo><abstract/><relatedItem type="host"><titleInfo><title>General 

### Making the data more easily readable

While XML is generally used to store computer-readable and -processable data,
BeautifulSoup is designed for working with HTML, which is designed for data and presentation.
BeautifulSoup thus has some nice features to make the data a bit easier to read.
As noted in [](#mods-ex01), each record is on a long line, which is difficult to read.
To help with this, the `.prettify()` function is useful.

Reusing the `first_mods` variable from above, the following code
displays that record in a more easily readable way with the prettify function.

In [11]:
print(first_mods.prettify())

<mods version="3.4" xmlns="http://www.loc.gov/mods/v3" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.loc.gov/mods/v3 http://www.loc.gov/standards/mods/v3/mods-3-4.xsd">
 <identifier>
  lcwaN0010234
 </identifier>
 <identifier invalid="yes" type="database id">
  85999
 </identifier>
 <identifier invalid="yes" type="database id">
  109353
 </identifier>
 <titleInfo>
  <title>
   Slate Magazine
  </title>
 </titleInfo>
 <language>
  <languageTerm authority="iso639-2b" type="code">
   eng
  </languageTerm>
 </language>
 <physicalDescription>
  <form authority="marcform">
   electronic
  </form>
  <internetMediaType>
   text/html
  </internetMediaType>
  <digitalOrigin>
   born digital
  </digitalOrigin>
 </physicalDescription>
 <targetAudience>
  general
 </targetAudience>
 <typeOfResource>
  text
 </typeOfResource>
 <genre authority="marcgt">
  web site
 </genre>
 <originInfo>
  <place>
   <placeTerm type="t

## Data Exploration and Quality Review

Working with the XML file as parsed markup data creates a data structure that allows Python to process and manipulate particular elements as data. The next sections illustrate various ways to work with an element's data. First, we explore the nature of XML _as data_, and then look at some of the functions useful in querying specific parts of the data.

Data in elements can often be isolated or extracted with the `.text` method. This is not always the case, however,
since some elements only wrap around other data elements or, as we saw earlier, may be blank.
It's worth looking, therefore, at how to extract the text data, if it is preesnt 

Beyond basic retrieval of specific elements, BeautifulSoup provides additional
searching and querying functions that can support
identifying and filtering elements that meet specific criteria.
For example, listing or looping through all of the `title` tags,
looking for particular attributes, or checking for valid identifiers.
Such querying can be useful to look broadly at the extent, quality, and shape of your data.[^advanced-py-fn]

[^advanced-py-fn]: More complex XML operations, like multiple namespaces, schema validation, and updating or writing data, will be discussed in [](/part02/xml-03-advanced-MODS.ipynb).

### Using the `.text` Method

Earlier, `.text` was used on a `title` element. That element is required to wrap a string.
However, many MODS elements are not like that: they hold only *more elements*.
In those cases, the text lives further down the tree.
The `physicalDescription` element is one of these. Take a look at its shape:

In [12]:
physDescrip = metadata.find('physicalDescription')

print(physDescrip.prettify())

<physicalDescription>
 <form authority="marcform">
  electronic
 </form>
 <internetMediaType>
  text/html
 </internetMediaType>
 <digitalOrigin>
  born digital
 </digitalOrigin>
</physicalDescription>



The `<physicalDescription>` tag contains no text of its own. In the above example,
this element contains three child elements: `form`, `internetMediaType`, and `digitalOrigin`.
Each of those subelements holds a string.

BeautifulSoup's `.text` searches *all* the way down the tree and joins together
every string it finds. Here it is applied to the parent element. The `repr()`
function is used so that the exact string is displayed, including any spaces or
line breaks:

In [13]:
print(repr(physDescrip.text))

'electronictext/htmlborn digital'


Take a close look at the above response: `'electronictext/htmlborn digital'`.
The string runs together three values into a single string with no spaces in between.
BeautifulSoup does not insert separators. It only returns exactly the characters
found in the file, and this file has no spaces or line breaks between the tags.

In this case, `.text` offers a quick way to check, "is there anything in here at all?"
On the other hand, it provides an unreliable way to build a value
that you may want to store or later display.
If the the parts matter, use `.get_text()` and supply a separator.

In [14]:
print(physDescrip.get_text(' -- ', strip=True))

electronic -- text/html -- born digital


If you know which elements you want, it is possible to select them by name
using the dot notation on each,
rather than flattening the whole branch. This keeps each value distinct:

In [15]:
for record in metadata.find_all('mods'):
    physDescrip = record.find('physicalDescription')
    print(physDescrip.form.text, '|', physDescrip.internetMediaType.text, '|', physDescrip.digitalOrigin.text)

electronic | text/html | born digital
electronic | text/html | born digital
electronic | text/html | born digital
electronic | text/html | born digital
electronic | text/html | born digital


:::{hint} Whitespace is data
The `<physicalDescription>` elements in this file are written without indentation,
so `.text` runs the values together. Had the file been "pretty printed" with each
child element on its own line, the same `.text` call would have returned
`'\nelectronic\ntext/html\nborn digital\n'`. In that case, the `\n` characters indicate
line breaks, which are produced by the file's formatting, not from anything BeautifulSoup added.
Either way, the text of a parent element reflects how the file was written,
which is why it should not be trusted as a clean value.
:::

### Looking for Sets of Elements with `.find_all()`

Previously we identified the first `title` element with dot notation and `.find()` to select the first `mods` element.
But what if you were looking for all of the occurences of a particular element?
For example, what if you wanted to find all of the `title` tags in a given record or collection?
The [MODS documentation](https://www.loc.gov/standards/mods/userguide/titleinfo.html#title)
defines a title as "a word, phrase, character, or group of characters that constitutes the chief title of a resource, i.e., the title normally used when citing the resource." This element is, furthermore, wrapped in a `titleInfo` element. If the MODS sample contains five full MODS records, it likely includes at least one title for each of these records. To see what titles are present, use BeautifulSoup's `.find_all()` function. The following loop first loops through each `mods` record, then shows the title elements in each:

In [16]:
count = 0 

for mods in metadata.find_all('mods'):
    count += 1
    print(mods.name,count)
    for title in mods.find_all('title'):
        print(title.name, title.text, title.parent)
    print('\n')

mods 1
title Slate Magazine <titleInfo><title>Slate Magazine</title></titleInfo>
title General News on the Internet Web Archive <titleInfo><title>General News on the Internet Web Archive</title></titleInfo>
title Serial and Government Publications Division <titleInfo><title>Serial and Government Publications Division</title></titleInfo>


mods 2
title Raw Story <titleInfo><title>Raw Story</title></titleInfo>
title General News on the Internet Web Archive <titleInfo><title>General News on the Internet Web Archive</title></titleInfo>
title Serial and Government Publications Division <titleInfo><title>Serial and Government Publications Division</title></titleInfo>


mods 3
title Huffington Post <titleInfo><title>Huffington Post</title></titleInfo>
title General News on the Internet Web Archive <titleInfo><title>General News on the Internet Web Archive</title></titleInfo>
title Serial and Government Publications Division <titleInfo><title>Serial and Government Publications Division</title>

It's now clear that each `mods` record contains three `title` elements,
each designating a slightly different aspect. The first element shows the name of the record.
The second shows the name of the overall web archive collection, in this case "General News on the Internet Web Archive".
the third shows the custodial unit at the Library of Congress, in this case the Serials Division for each record.

### Other `.find_all()` filters

The `.find_all()` function accepts many types of filters beyond element names,
including regular expressions, lists, and even custom functions, as detailed here in the BeautifulSoup documentation.
The function can also be used to return any element when it is provided the argument `True`.
The following demonstrates this usage to get a list of all the tags in the data
(using `True` to demonstrate the existence of each tag in the file). Note also the use of the `limit` argument,
which in this course returns only 10 instances. In this demonstration it is not necessary to return the the entire list of elements in the document.

In [17]:
for element in metadata.find_all(True, limit=10):
    print(element.name)

modsCollection
mods
identifier
identifier
identifier
titleInfo
title
language
languageTerm
physicalDescription


```{exercise} How many element tags are in this MODS collection?
:label: ex-bs4-count-el
How many element tags are in the five sample MODS file?
```

In [18]:
count = 0

for element in metadata.find_all(True):
    count += 1

print(count)

247


```{solution} ex-bs4-count-el
:class: dropdown
There are **247 elements** in the file.
```

### Examining element attributes

Any individual element in an XML tree might contain attributes, so
it is useful to know how to look for this data.
When the parser finds attributes, it stores them in a dictionary-like object, which can be referenced from the source element and is indicated with curly braces (`{}`).

Take a look back at [the MODS excerpt](#mods-ex01). Notice that the `languageTerm` element contains two attributes, `authority` and `type`.
A construction like the following block uses the `.find()` function to
look for the first `languageTerm` element, assigns it to the `langElem` variable,
then uses the methods `.name`, `attrs`, and `.text` methods to print out the values.

In [19]:
langElem = metadata.find('languageTerm')

print('Tag:\t\t\t',langElem.name)
print('Attributes and values:\t',langElem.attrs)
print('Tag contents:\t\t',langElem.text)

Tag:			 languageTerm
Attributes and values:	 {'authority': 'iso639-2b', 'type': 'code'}
Tag contents:		 eng


Working with attributes is an important task. It can be used, for example, to conduct
quality review or assess the metadata.
For example, the MODS definition of `languageTerm` does not require the use of an authority source to describe a resource's langauge, but it may be required by local metadata practices.
A construction like the following, which uses BeautifulSoup's `.has_attr()` function, could be used to check for language elements that don't have and `@authority` attribute.

In [20]:
for langTerm in metadata.find_all('languageTerm'):
    if not langTerm.has_attr('authority'):
        print('Missing authority attribute:',langTerm)
    elif langTerm['authority'] != 'iso639-2b':
        print('Unexepected authority value:',langTerm)
    else:
        print('This one looks fine!')

This one looks fine!
This one looks fine!
This one looks fine!
This one looks fine!
This one looks fine!
This one looks fine!
Unexepected authority value: <languageTerm authority="marc" type="code">eng</languageTerm>
This one looks fine!
Missing authority attribute: <languageTerm type="code">eng</languageTerm>
This one looks fine!


### The `.get()` function for values

While it may seem easiest look directly for the authority value using
python's dictionary key name notation (e.g., `languageTerm['authority']`),
it is generally best to avoid this. It is more idiomatic to use a function that can be used or reused as a data object, and this is the purpose of using Python tools to parse an XML document. Instead, BeautifulSoup's `.get()` function is preferable since it creates a data object that can be manipulated rather than a standalone string or `None` value. For example, the following is another way to identify `@authority` attributes that do not use the `iso639-2b` vocabulary.
This data is much more difficult to use beyond simple confirmation of its presence without the get function.

In [21]:
for langTerm in metadata.find_all('languageTerm'):
    authority = langTerm.get('authority')
    if authority == 'iso639-2b':
        print('Valid',authority)
    else:
        print('Found invalid language authority:',authority)

Valid iso639-2b
Valid iso639-2b
Valid iso639-2b
Valid iso639-2b
Valid iso639-2b
Valid iso639-2b
Found invalid language authority: marc
Valid iso639-2b
Found invalid language authority: None
Valid iso639-2b


### Listing attribute values

The above output demonstrates that some elements do not have an `@authority` attribute, and also that multiple authorities are in use.
It might, therefore, be useful to identify all of the unique values that appear in authority attributes,
which will help to determine which controlled vocabularies are in use.
The following approach creates this information using a Python `set`, which is a list of unique values. Although a list could also work, it is quicker to search a set and it automatically hedges against duplicate values.

In [22]:
langAuthorities = set()

for lang in metadata.find_all('languageTerm'):
    authority = lang.get('authority')
    if authority:
        langAuthorities.add(authority)

for authority in sorted(langAuthorities):
    print(authority)

iso639-2b
marc


### Counting Attribute Values

Add value to the above by tallying the number of times
each value is used. This could be a useful way to identify
outliers, outdated values, or just for general interest.

:::{attention} Introducing the `Counter()` function
The following cell imports a new Python module.
Notice the first line, which reads `from collections import Counter`.
This enables the use of the `Counter()` function,
which is a reusable tool from from Python's `collections` module.
The module provides useful tools helpful for working with sets, or "collections" of data stored in compound data types (like tuples, dictionaries, and sets), like enumerating unique values, counting occurences, sorting complex lists, and more.

Read the `collections` documentation at <https://docs.python.org/3/library/collections.html>.
:::

In [23]:
from collections import Counter

authorityCounts = Counter(
    lang.get('authority') for lang in metadata.find_all('languageTerm') if lang.get('authority', 'NO AUTHORITY')
)

print(authorityCounts.most_common())

[('iso639-2b', 8), ('marc', 1), (None, 1)]


The above uses a `Counter()` function from the useful `collections` library. Inside the `Counter`, a list comprehension simplifies a filter expression. The comprehension uses `for` to identify all of the `languageTerm` elements in the full `metadata` data object, filters them using an `if` statement to request the `@authority` elements (the `'NO AUTHORITY'` argument requests even blank values to be tallied), and then provides the responses to the Counter.
The Counter stores each unique value and notes how many times it occurs, which is called with the `.most_common()` function. 

```{exercise} Create a list of the values of type attributes
:label: ex-bs4-type-counter
If you look through the five MODS records, you will note many `@type`
attributes. As demonstrated with the authority attributes above,
create a list and count of values used in the type attribute
```

``````{solution} ex-bs4-type-counter
:label: ex-bs4-type-counter-solution
:class: dropdown
The python code would be similar to the Counter loop previously created for the language authorities.
The response is not run here, but a possible code solution (assuming that the Counter function is already imported) could be:

```{code-block}python
typeCounts = Counter(
    typeAttrib.get('authority') for typeAttrib in metadata.find_all(True) if typeAttrib.get('type', 'NO AUTHORITY')
)

print(typeCounts.most_common())
```

``````

## Close the file

Finally, don't forget to close the file. Subsequent examples use the `with open() as variable` method of opening a file.
Above, however, the `open` method is not closed until `close` is called.
While this isn't going to cause major problems, it does mean the file is open as data (until closed), and some systems will not allow for other tools to use or modify the file while it is open.

In [24]:
file.close()

The rest of the cells in this section, and further on in the book, use the
more efficient `with open(file) as data:` construction available in python.

## Combining parsing, `.find_all()`, and looping

While the previous examples intended to illustrate individual operations
on their own, it can be most effective to combine functions together to create
analyze your data, create small reports, or to isolate different sections of your data
for closer inspection. The following cell looks
through each `mods` record, then prints the tag name, `title` element, and the `form` element. After printing those three pieces of data to the output, the loop counts the iterations and confirms there are five elements.

In [25]:
record_count = 0

with open(MODS_collection, 'r') as mods_records:
    metadata = BeautifulSoup(mods_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        print(mods.name, mods.title, mods.form)
        record_count += 1

print(record_count)

mods <title>Slate Magazine</title> <form authority="marcform">electronic</form>
mods <title>Raw Story</title> <form authority="marcform">electronic</form>
mods <title>Huffington Post</title> <form authority="marcform">electronic</form>
mods <title>BuzzFeed</title> <form authority="marcform">electronic</form>
mods <title>Drudge Report</title> <form authority="marcform">electronic</form>
5


## Exercises / Challenges

:::{warning} Adapt the Below as Additional Examples or Problems

1. A dot-notation examples
1. Looking through identifiers, titles, or whatever element is desired
1. Some elements contain text as data, some only contain other data elements. 
1. Look at the `subject tags`, how would you filter by lcsh authority? Work with keyword filters by attribute
1. Count some of the other elements with the Counter tool?
:::



#### 2. Extract Item Identifiers

Activity 2: Each individual metadata record has at least one `<identifier>` element; this element is used to include a reference to the item, such as a URI, or another identifier that a system may use to locate an item. Using a loop similar to the example above, how would you print each record's identifier(s)?  

In [26]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        for identifier in mods.find_all('identifier'):
            print(identifier.name, identifier.text)

NameError: name 'MODS_file' is not defined

There are clearly different types of identifiers here, and when we check the identifier attributes, 
it is clear that some of these will be more useful than others. Below, use the `.attrs` method to see the dictionary that each element carries:

In [ ]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        for identifier in mods.find_all('identifier'):
            print(identifier.name, identifier.text, identifier.attrs)

Since some of the elements do not have attributes, we need a try-except loop to look at each dictionary, 
and to generate a "Blank" value for elements without attributes:

In [ ]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        for identifier in mods.find_all('identifier'):
            tag = identifier.name
            content = identifier.text
            try:
                type_ = identifier.attrs['type']
            except:
                type_ = "Blank type"
            print(tag, content, type_)

Finally, let's print a clean list of only the URI identifiers:

In [ ]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        for identifier in mods.find_all('identifier', type="uri"):
            print(identifier.attrs['type'], identifier.text)

Now, we could try that another way using the `lxml` XML library directly:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for identifier in xml_records.findall('.//mods:identifier', namespaces=ns): 
    element = identifier
    print(element.tag, element.text, element.attrib)

And, filter to identify only the URI elements...

Using an XPath expression `etree`, you can filter directly for identifiers that
have a `type` attribute with the `uri` value:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for identifier in xml_records.findall('.//mods:identifier[@type="uri"]', namespaces=ns): 
    element = identifier
    attribs = element.attrib
    type = attribs.get('type')
    print(element.tag, type, element.text)

#### 3. Extract the record titles 

Activity 3: Each individual metadata record has `<titleInfo>` that contains at least one `<title>` element; this element is used to identyify an item's title. Using a loop similar to the example above, how would you print each record's title(s)?  

In [ ]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        for title in mods.find_all('title'):
            print(title.name, title.find_parent())

Similarly, `lxml` may be used with the `.getparent()` function. Note that this is not possible with the ElementTree module, only with the lxml module.

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)
metadata = xml_records.getroot()

for mods in metadata.findall('.//mods:mods', namespaces=ns):
    for title in mods.findall('.//mods:title', namespaces=ns):
        parent = title.getparent()
        print(title.tag, parent.tag)

Note above that the `.find_parent()` (BS4) or `.getparent()` (lxml) methods can be used to look "up" the tree, 
in this case displaying the parent element of the `title` element.

A similar result can be produced using the `lxml` library directly. In this case, 
the `.findall()` method is similar, but notice that the request can be given
using XPATH references and while specifying namespaces:

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)

for titleInfo in xml_records.findall('.//mods:title', namespaces=ns): 
    element = titleInfo
    print(element.text)

In [ ]:
for title in xml_records.findall('.//mods:title', namespaces=ns):
    print(title.text)

#### 4. Extract only the Main Titles

Activity 4: Notice in the previous activity that even though we are working with only five
records, there are well more than five titles. Each of these records has multiple `title` elements,
some of which are for `relatedItem` elements. If we want only the main titles, use the `.find()` function 
to search for only the first instance and print out only the main title. An alternative way to do this in `lxml`
is to use a more specific XPath selector.

In [ ]:
#BS4
with open(MODS_file, 'r') as xml_records:
    metadata = BeautifulSoup(xml_records, 'lxml-xml')
    for mods in metadata.find_all('mods'):
        title = mods.find('title')
        print(title.name, title.text)

Try `lxml` to use an XPath request. (Note: you can also use the `.find()` method in `lxml` to return only the first result.)

In [ ]:
#lxml
xml_records = etree.parse(MODS_file)
metadata = xml_records.getroot()

for title in metadata.findall('.//mods:mods/mods:titleInfo/mods:title', namespaces=ns): 
    print(title.text, title.tag)

Above, the query specifically asks for the `title` elements that are direct 
child elements of a `titleInfo` element, which is a child of the `mods` element. 
This is necessary to filter out any `titleInfo` elements that are actually under a
`relatedItem` element. With less specificity, the query will return numerous elements 
that "related" but not the title of the actual item: 

:::{warning} TRY ... Count with the Counter

Instead of as above, try an "Element census" with `Counter` — a natural extension of cell 24 and a much better payoff than the bare tag list. Example:

```python
from collections import Counter
Counter(tag.name for tag in metadata.find_all(True)).most_common()
```
NB: this uses a list comprehension

This addresses the question, "Which elements does this collection actually use, and how often?" is a real first-look question, and it sets up the QC section later. Can also potentially create a table or something that shows which tags occur and how often, ranked by occurence? 